**Chapter 15 – Transformers for Natural Language Processing and Chatbots**

_This notebook contains all the sample code and solutions to the exercises in chapter 15._

<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/ageron/handson-mlp/blob/main/15_transformers_for_nlp_and_chatbots.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
  <td>
    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/ageron/handson-mlp/blob/main/15_transformers_for_nlp_and_chatbots.ipynb"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" /></a>
  </td>
</table>

# Setup

This project requires Python 3.10 or above:

In [2]:
import sys

assert sys.version_info >= (3, 10)

Are we using Colab or Kaggle?

In [3]:
IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "kaggle_secrets" in sys.modules

If using Colab, the TorchMetrics library is not pre-installed so we must install it manually:

In [4]:
if IS_COLAB:
    %pip install -qU torchmetrics

if IS_KAGGLE:
    %pip install -qU transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 50.2 MB/s eta 0:00:00


We also need PyTorch ≥ 2.6.0:

In [5]:
from packaging.version import Version
import torch

assert Version(torch.__version__) >= Version("2.6.0")

This chapter can be very slow without a hardware accelerator, so if we can find one, let's use it:

In [6]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cuda'

Let's issue a warning if there's no hardware accelerator available:

In [7]:
if device == "cpu":
    print("Neural nets can be very slow without a hardware accelerator.")
    if IS_COLAB:
        print("Go to Runtime > Change runtime and select a GPU hardware "
              "accelerator.")
    if IS_KAGGLE:
        print("Go to Settings > Accelerator and select GPU.")

As we did in earlier chapters, let's define the default font sizes to make the figures prettier:

In [8]:
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

Let's use the same `evaluate_tm()` and `train()` functions as in the previous chapters:

In [9]:
import torchmetrics

def evaluate_tm(model, data_loader, metric):
    model.eval()
    metric.reset()
    with torch.no_grad():
        for X_batch, y_batch in data_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            metric.update(y_pred, y_batch)
    return metric.compute()

def train(model, optimizer, loss_fn, metric, train_loader, valid_loader,
          n_epochs, patience=2, factor=0.5, epoch_callback=None):
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", patience=patience, factor=factor)
    history = {"train_losses": [], "train_metrics": [], "valid_metrics": []}
    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()
        if epoch_callback is not None:
            epoch_callback(model, epoch)
        for index, (X_batch, y_batch) in enumerate(train_loader):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)
            total_loss += loss.item()
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            metric.update(y_pred, y_batch)
            train_metric = metric.compute().item()
            print(f"\rBatch {index + 1}/{len(train_loader)}", end="")
            print(f", loss={total_loss/(index+1):.4f}", end="")
            print(f", {train_metric=:.2%}", end="")
        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)
        val_metric = evaluate_tm(model, valid_loader, metric).item()
        history["valid_metrics"].append(val_metric)
        scheduler.step(val_metric)
        print(f"\rEpoch {epoch + 1}/{n_epochs},                      "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.2%}, "
              f"valid metric: {history['valid_metrics'][-1]:.2%}")
    return history

As we will build and download pretty big models, we will need to free the GPU RAM regularly to avoid running out of space. For this, we will delete the models and tensors as we go, using the `del_vars()` function below. It deletes the given variables (if they exist) then calls Python's garbage collector, and it also calls `torch.cuda.empty_cache()` if we're using a CUDA GPU:

In [10]:
import gc

def del_vars(variable_names=[]):
    for name in variable_names:
        try:
            del globals()[name]
        except KeyError:
            pass  # ignore variables that have already been deleted
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

**WARNING**: When running a Jupyter/Colab notebook, the output of each cell gets saved in the `Out` dictionary, so if the output of a cell is a large model or tensor, then it's not enough to delete the variable, you must also delete the output from the `Out` dictionary (e.g., by clearing the whole dictionary with `Out.clear()`).

## Prepare the NMT dataset
This section downloads and prepares the NMT dataset exactly like in the previous chapter.

In [11]:
import torch.nn as nn
from datasets import load_dataset
from torch.utils.data import random_split, DataLoader
import tokenizers

In [12]:
nmt_original_valid_set, nmt_test_set = load_dataset(
    path="ageron/tatoeba_mt_train", name="eng-spa",
    split=["validation", "test"])
split = nmt_original_valid_set.train_test_split(train_size=0.8, seed=42)
nmt_train_set, nmt_valid_set = split["train"], split["test"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.34k [00:00<?, ?B/s]

eng-spa/validation-00000-of-00001.parque(…):   0%|          | 0.00/7.85M [00:00<?, ?B/s]

eng-spa/test-00000-of-00001.parquet:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/197299 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/24514 [00:00<?, ? examples/s]

In [13]:
nmt_train_set[0]

{'source_text': 'Tom tried to break up the fight.',
 'target_text': 'Tom trató de disolver la pelea.',
 'source_lang': 'eng',
 'target_lang': 'spa'}

In [14]:


ds = load_dataset("bentrevett/multi30k")
Xtrain, Xvalid, Xtest = ds["train"], ds["validation"], ds["test"]

print(Xtrain[0])   # {'en': 'Two young, White males are ...', 'de': 'Zwei junge weiße Männer ...'}

# English -> German: source = en, target = de
nmt_train_set = [(row["en"], row["de"]) for row in Xtrain]
nmt_valid_set = [(row["en"], row["de"]) for row in Xvalid]

README.md:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/4.60M [00:00<?, ?B/s]

val.jsonl:   0%|          | 0.00/164k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/156k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/29000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1014 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

{'en': 'Two young, White males are outside near many bushes.', 'de': 'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'}


In [15]:
nmt_train_set[0][1]

'Zwei junge weiße Männer sind im Freien in der Nähe vieler Büsche.'

In [16]:
def train_eng_spa():  # a generator function to iterate over all training text
    for pair in nmt_train_set:
        yield pair[0]
        yield pair[1]

max_length = 500
vocab_size = 10_000
nmt_tokenizer_model = tokenizers.models.BPE(unk_token="<unk>")
nmt_tokenizer = tokenizers.Tokenizer(nmt_tokenizer_model)
nmt_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
nmt_tokenizer.enable_truncation(max_length=max_length)
nmt_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace()
nmt_tokenizer_trainer = tokenizers.trainers.BpeTrainer(
    vocab_size=vocab_size, special_tokens=["<pad>", "<unk>", "<s>", "</s>"])
nmt_tokenizer.train_from_iterator(train_eng_spa(), nmt_tokenizer_trainer)

In [17]:
from collections import namedtuple

fields = ["src_token_ids", "src_mask", "tgt_token_ids", "tgt_mask"]
class NmtPair(namedtuple("NmtPairBase", fields)):
    def to(self, device):
        return NmtPair(self.src_token_ids.to(device), self.src_mask.to(device),
                       self.tgt_token_ids.to(device), self.tgt_mask.to(device))

In [36]:
def nmt_collate_fn(batch):
    src_texts = [pair[0] for pair in batch]
    tgt_texts = [pair[1] for pair in batch]
    src_encodings = nmt_tokenizer.encode_batch(src_texts)
    tgt_encodings = nmt_tokenizer.encode_batch(tgt_texts)
    src_token_ids = torch.tensor([enc.ids for enc in src_encodings])
    tgt_token_ids = torch.tensor([enc.ids for enc in tgt_encodings])
    src_mask = torch.tensor([enc.attention_mask for enc in src_encodings])
    tgt_mask = torch.tensor([enc.attention_mask for enc in tgt_encodings])
    inputs = NmtPair(src_token_ids, src_mask,
                     tgt_token_ids[:, :-1], tgt_mask[:, :-1])
    labels = tgt_token_ids[:, 1:]
    return inputs, labels

batch_size = 64
nmt_train_loader = DataLoader(nmt_train_set, batch_size=batch_size,
                              collate_fn=nmt_collate_fn, shuffle=True)
nmt_valid_loader = DataLoader(nmt_valid_set, batch_size=batch_size,
                              collate_fn=nmt_collate_fn)
nmt_test_loader = DataLoader(nmt_test_set, batch_size=batch_size,
                             collate_fn=nmt_collate_fn)

# Attention Is All You Need: The Original Transformer Architecture

## Positional encodings

In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PositionalEmbedding(nn.Module):
    def __init__(self, max_length, embed_dim, dropout=0.1):
        super().__init__()
        self.pos_embed = nn.Parameter(torch.randn(max_length, embed_dim) * 0.02)
        self.dropout = nn.Dropout(dropout)

    def forward(self, X):
        return self.dropout(X + self.pos_embed[:X.size(1)])

In [20]:
embed_dim = 512
pos_embedding = PositionalEmbedding(max_length, embed_dim)
embeddings = torch.randn(256, 500, 512)
embeddings_with_pos = pos_embedding(embeddings)
embeddings_with_pos.shape

torch.Size([256, 500, 512])

## Multi-Head Attention (MHA)

Let's implement a custom multi-head attention module to better understand how it works:

In [21]:
class MultiheadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.h = num_heads
        self.d = embed_dim // num_heads
        self.q_proj = nn.Linear(embed_dim, embed_dim)
        self.k_proj = nn.Linear(embed_dim, embed_dim)
        self.v_proj = nn.Linear(embed_dim, embed_dim)
        self.out_proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, X):
        return X.view(X.size(0), X.size(1), self.h, self.d).transpose(1, 2)

    def forward(self, query, key, value, attn_mask=None, key_padding_mask=None):
        q = self.split_heads(self.q_proj(query))  # (B, h, Lq, d)
        k = self.split_heads(self.k_proj(key))  # (B, h, Lk, d)
        v = self.split_heads(self.v_proj(value))  # (B, h, Lv, d) with Lv=Lk
        scores = q @ k.transpose(2, 3) / self.d**0.5  # (B, h, Lq, Lk)

        # Masking support:
        if attn_mask is not None:
            scores = scores.masked_fill(attn_mask, -torch.inf)  # (B, h, Lq, Lk)
        if key_padding_mask is not None:
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)  # (B, 1, 1, Lk)
            scores = scores.masked_fill(mask, -torch.inf)  # (B, h, Lq, Lk)

        weights = scores.softmax(dim=-1)  # (B, h, Lq, Lk)
        Z = self.dropout(weights) @ v  # (B, h, Lq, d)
        Z = Z.transpose(1, 2)  # (B, Lq, h, d)
        Z = Z.reshape(Z.size(0), Z.size(1), self.h * self.d)  # (B, Lq, h × d)
        return (self.out_proj(Z), weights)  # (B, Lq, h × d)

The shapes are indicated with the following abbreviations:
* `B`: batch size
* `h`: number of heads
* `Lq`: max query length (in the batch)
* `Lk`: max key length (in the batch)
* `Lv`: max value length (in the batch). Values must have the same length as keys, so Lv=Lk.
* `d`: embed_dim // h

Note that PyTorch's `nn.MultiheadAttention` layer groups the three input projection layer parameters into just two parameters: `in_proj_weight` (shape (_embed_dim_ * 3, _embed_dim_) and `in_proj_bias` (shape (_embed_dim_ * 3,)).

To get the same results with the `nn.MultiheadAttention` module, you must set `batch_first=True` in the constructor. Also, if you don't need the weights, you can set `need_weights=False` when calling the `forward()` method: in many cases, this allows the module to optimize the computations, giving the same attention output, but the weights will be set to `None`.

## Building the rest of the Transformer

In [22]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiheadAttention(d_model, nhead, dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, src, src_mask=None, src_key_padding_mask=None):
        attn, _ = self.self_attn(src, src, src, attn_mask=src_mask,
                                 key_padding_mask=src_key_padding_mask)
        Z = self.norm1(src + self.dropout(attn))
        ff = self.dropout(self.linear2(self.dropout(self.linear1(Z).relu())))
        return self.norm2(Z + ff)

In [23]:
class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        self.self_attn = MultiheadAttention(d_model, nhead, dropout)
        self.multihead_attn = MultiheadAttention(d_model, nhead, dropout)
        self.dropout = nn.Dropout(dropout)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None,
                tgt_key_padding_mask=None, memory_key_padding_mask=None):
        attn1, _ = self.self_attn(tgt, tgt, tgt,
                                  attn_mask=tgt_mask,
                                  key_padding_mask=tgt_key_padding_mask)
        Z = self.norm1(tgt + self.dropout(attn1))
        attn2, _ = self.multihead_attn(Z, memory, memory, attn_mask=memory_mask,
                                       key_padding_mask=memory_key_padding_mask)
        Z = self.norm2(Z + self.dropout(attn2))
        ff = self.dropout(self.linear2(self.dropout(self.linear1(Z).relu())))
        return self.norm3(Z + ff)

The following modules are straightforward and were left as an exercise to the reader:

In [24]:
from copy import deepcopy

class TransformerEncoder(nn.Module):
    def __init__(self, encoder_layer, num_layers, norm=None):
        super().__init__()
        self.layers = nn.ModuleList([deepcopy(encoder_layer)
                                     for _ in range(num_layers)])
        self.norm = norm

    def forward(self, src, mask=None, src_key_padding_mask=None):
        Z = src
        for layer in self.layers:
            Z = layer(Z, mask, src_key_padding_mask)
        if self.norm is not None:
            Z = self.norm(Z)
        return Z

In [25]:
class TransformerDecoder(nn.Module):
    def __init__(self, decoder_layer, num_layers, norm=None):
        super().__init__()
        self.layers = nn.ModuleList([deepcopy(decoder_layer)
                                     for _ in range(num_layers)])
        self.norm = norm

    def forward(self, tgt, memory, tgt_mask=None, memory_mask=None,
                tgt_key_padding_mask=None, memory_key_padding_mask=None):
        Z = tgt
        for layer in self.layers:
            Z = layer(Z, memory, tgt_mask, memory_mask,
                      tgt_key_padding_mask, memory_key_padding_mask)
        if self.norm is not None:
            Z = self.norm(Z)
        return Z

In [26]:
class Transformer(nn.Module):
    def __init__(self, d_model=512, nhead=8, num_encoder_layers=6,
                 num_decoder_layers=6, dim_feedforward=2048, dropout=0.1):
        super().__init__()
        encoder_layer = TransformerEncoderLayer(d_model, nhead, dim_feedforward,
                                                dropout)
        norm1 = nn.LayerNorm(d_model)
        self.encoder = TransformerEncoder(encoder_layer, num_encoder_layers,
                                          norm1)
        decoder_layer = TransformerDecoderLayer(d_model, nhead, dim_feedforward,
                                                dropout)
        norm2 = nn.LayerNorm(d_model)
        self.decoder = TransformerDecoder(decoder_layer, num_decoder_layers,
                                          norm2)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None, memory_mask=None,
                src_key_padding_mask=None, tgt_key_padding_mask=None,
                memory_key_padding_mask=None):
        memory = self.encoder(src, src_mask, src_key_padding_mask)
        output = self.decoder(tgt, memory, tgt_mask, memory_mask,
                              tgt_key_padding_mask, memory_key_padding_mask)
        return output

# Building an English-to-Spanish Transformer

In [27]:
class NmtTransformer(nn.Module):
    def __init__(self, vocab_size, max_length, embed_dim=512, pad_id=0,
                 num_heads=8, num_layers=6, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.pos_embed = PositionalEmbedding(max_length, embed_dim, dropout)
        self.transformer = nn.Transformer(
            embed_dim, num_heads, num_encoder_layers=num_layers,
            num_decoder_layers=num_layers, batch_first=True)
        self.output = nn.Linear(embed_dim, vocab_size)

    def forward(self, pair):
        src_embeds = self.pos_embed(self.embed(pair.src_token_ids))
        tgt_embeds = self.pos_embed(self.embed(pair.tgt_token_ids))
        src_pad_mask = ~pair.src_mask.bool()
        tgt_pad_mask = ~pair.tgt_mask.bool()
        size = [pair.tgt_token_ids.size(1)] * 2
        full_mask = torch.full(size, True, device=tgt_pad_mask.device)
        causal_mask = torch.triu(full_mask, diagonal=1)
        out_decoder = self.transformer(src_embeds, tgt_embeds,
                                       src_key_padding_mask=src_pad_mask,
                                       memory_key_padding_mask=src_pad_mask,
                                       tgt_mask=causal_mask, #tgt_is_causal=True,
                                       tgt_key_padding_mask=tgt_pad_mask)
        return self.output(out_decoder).permute(0, 2, 1)

In [28]:
torch.triu(torch.full((5, 5), True), diagonal=1)

tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])

In [29]:
nn.Transformer.generate_square_subsequent_mask(5)

tensor([[0., -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0.]])

In [30]:
torch.manual_seed(42)
nmt_tr_model = NmtTransformer(vocab_size, max_length, embed_dim=128, pad_id=0,
                              num_heads=4, num_layers=2, dropout=0.1).to(device)
if device == "mps":
    # WORKAROUND: on MPS devices, we use our custom Transformer because the
    # nn.Transformer module explodes during training, see PyTorch issue #141287
    nmt_tr_model.transformer = Transformer(
        d_model=128, nhead=4, num_encoder_layers=2, num_decoder_layers=2).to(device)

n_epochs = 20
xentropy = nn.CrossEntropyLoss(ignore_index=0)  # ignore <pad> tokens
optimizer = torch.optim.NAdam(nmt_tr_model.parameters())
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=vocab_size)
accuracy = accuracy.to(device)

history = train(nmt_tr_model, optimizer, xentropy, accuracy,
                nmt_train_loader, nmt_valid_loader, n_epochs)

Batch 454/454, loss=4.4186, train_metric=15.86%

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


Epoch 1/20,                      train loss: 4.4186, train metric: 15.86%, valid metric: 17.60%
Epoch 2/20,                      train loss: 3.1814, train metric: 21.68%, valid metric: 22.33%
Epoch 3/20,                      train loss: 2.6346, train metric: 24.61%, valid metric: 23.40%
Epoch 4/20,                      train loss: 2.3039, train metric: 26.43%, valid metric: 25.49%
Epoch 5/20,                      train loss: 2.0840, train metric: 27.65%, valid metric: 26.29%
Epoch 6/20,                      train loss: 1.9263, train metric: 28.51%, valid metric: 26.47%
Epoch 7/20,                      train loss: 1.8023, train metric: 29.30%, valid metric: 27.11%
Epoch 8/20,                      train loss: 1.7074, train metric: 29.89%, valid metric: 27.10%
Epoch 9/20,                      train loss: 1.6287, train metric: 30.66%, valid metric: 27.53%
Epoch 10/20,                      train loss: 1.5652, train metric: 30.98%, valid metric: 27.40%
Epoch 11/20,                      train

In [31]:
torch.save(nmt_tr_model.state_dict(), "my_nmt_tr_model.pt")

In [38]:
def translate(model, src_text, max_length=20, pad_id=0, eos_id=3):
    tgt_text = ""
    token_ids = []
    for index in range(max_length):
        batch, _ = nmt_collate_fn([0],
                                    [1])
        with torch.no_grad():
            Y_logits = model(batch.to(device))
            Y_token_ids = Y_logits.argmax(dim=1)  # find the best token IDs
            next_token_id = Y_token_ids[0, index]  # take the last token ID

        next_token = nmt_tokenizer.id_to_token(next_token_id)
        tgt_text += " " + next_token
        if next_token_id == eos_id:
            break
    return tgt_text

In [40]:
def translate(model, src_text, max_length=20, pad_id=0, eos_id=3):
    tgt_text = ""
    for index in range(max_length):
        batch, _ = nmt_collate_fn([(src_text, tgt_text)])   # one list of (src, tgt) pairs
        with torch.no_grad():
            Y_logits = model(batch.to(device))
            Y_token_ids = Y_logits.argmax(dim=1)            # (1, seq_len) token ids
            next_token_id = Y_token_ids[0, index].item()    # prediction at the last position
        if next_token_id == eos_id:
            break
        next_token = nmt_tokenizer.id_to_token(next_token_id)
        tgt_text += " " + next_token
    return tgt_text.strip()

In [42]:
def translate(model, src_text, max_length=20, eos_id=3):
    # encode the source once (the tokenizer adds BOS/EOS just like in training)
    src_enc = nmt_tokenizer.encode(src_text)
    src_ids  = torch.tensor([src_enc.ids], device=device)            # (1, src_len) Long
    src_mask = torch.tensor([src_enc.attention_mask], device=device)

    bos_id = src_enc.ids[0]          # tokenizer prepends BOS to every sequence
    tgt_ids = [bos_id]               # decoder starts with just BOS
    for _ in range(max_length):
        tgt_in   = torch.tensor([tgt_ids], device=device)            # (1, cur_len) Long
        tgt_mask = torch.ones_like(tgt_in)                           # no padding in a single seq
        batch = NmtPair(src_ids, src_mask, tgt_in, tgt_mask)
        with torch.no_grad():
            logits  = model(batch)                                   # (1, vocab, cur_len)
            next_id = logits[0, :, -1].argmax().item()               # next token = last position
        if next_id == eos_id:
            break
        tgt_ids.append(next_id)
    return nmt_tokenizer.decode(tgt_ids[1:])                         # drop BOS, return text

In [47]:
nmt_tr_model.eval()
translate(nmt_tr_model, " ")

'Hund Hund Hund Hund Hund Hund Hund Hund Hund Hund Hund Hund Hund Hund Hund Hund Hund Hund Hund Hund'

Let's free some GPU RAM:

In [ ]:
del accuracy, history, nmt_test_set, nmt_tokenizer
del nmt_tokenizer_model, nmt_test_loader, nmt_train_loader, nmt_valid_loader
del nmt_tr_model, nmt_train_set, nmt_valid_set, optimizer
del pos_embedding, xentropy
free_vram(device)

# Encoder-Only Transformers for Natural Language Understanding

Pretraining BERT with MLM on the WikiText dataset, using the Transformers library:

In [ ]:
from transformers import BertConfig, BertForMaskedLM, BertTokenizerFast

bert_tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
config = BertConfig(  # adapt to training budget, and dataset size & complexity
    vocab_size=bert_tokenizer.vocab_size, hidden_size=128, num_hidden_layers=2,
    num_attention_heads=4, intermediate_size=512, max_position_embeddings=128)
bert = BertForMaskedLM(config)

In [ ]:
from datasets import load_dataset

def tokenize(example, tokenizer=bert_tokenizer):
    return tokenizer(example["text"], truncation=True, max_length=128,
                     padding="max_length")

mlm_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
mlm_dataset = mlm_dataset.map(tokenize, batched=True)

In [ ]:
from transformers import Trainer, TrainingArguments
from transformers import DataCollatorForLanguageModeling

args = TrainingArguments(output_dir="./my_bert", num_train_epochs=5,
                         per_device_train_batch_size=16,
                         report_to="none")  # to avoid requiring W&B or TB
mlm_collator = DataCollatorForLanguageModeling(bert_tokenizer, mlm=True,
                                               mlm_probability=0.15)
trainer = Trainer(model=bert, args=args, train_dataset=mlm_dataset,
                  data_collator=mlm_collator)
trainer_output = trainer.train()

In [ ]:
from transformers import pipeline

torch.manual_seed(42)
fill_mask = pipeline("fill-mask", model=bert, tokenizer=bert_tokenizer)
top_predictions = fill_mask("The capital of [MASK] is Rome.")
top_predictions[0]

Before we continue, let's free some GPU RAM:

In [ ]:
del_vars(["bert_tokenizer", "config", "bert", "mlm_dataset", "args",
          "mlm_collator", "trainer", "trainer_output", "fill_mask",
          "top_predictions"])

In [ ]:
from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")
inputs = tokenizer(["I like soccer", "Hello, World!"],
                   padding=True, return_tensors="pt")
outputs = model(**inputs)
cls_embedding = outputs.last_hidden_state[:, 0, :]
cls_embedding.shape

In [ ]:
del tokenizer, model, inputs, outputs, cls_embedding
free_vram(device)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
sentences = ["She's shopping", "She bought some shoes", "She's working"]
embeddings = model.encode(sentences, convert_to_tensor=True)
similarities = model.similarity(embeddings, embeddings)

In [ ]:
similarities

Let's free some GPU RAM:

In [ ]:
del_vars(["model", "sentences", "embeddings", "similarities"])

# Decoder-Only Transformers

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "gpt2"
gpt2_tokenizer = AutoTokenizer.from_pretrained(model_id)
gpt2 = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", dtype="auto")

In [ ]:
def generate(model, tokenizer, prompt, max_new_tokens=50, **generate_kwargs):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             pad_token_id=tokenizer.eos_token_id,
                             **generate_kwargs)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
prompt = "Scientists found a talking unicorn today. Here's the full story:"
generate(gpt2, gpt2_tokenizer, prompt)

In [ ]:
torch.manual_seed(42)
generate(gpt2, gpt2_tokenizer, prompt, do_sample=True)

In [ ]:
torch.manual_seed(42)
generate(gpt2, gpt2_tokenizer, prompt, do_sample=True, top_p=0.6)

In [ ]:
DEFAULT_TEMPLATE = "Capital city of France = Paris\nCapital city of {country} ="

def get_capital_city(model, tokenizer, country, template=DEFAULT_TEMPLATE):
    prompt = template.format(country=country)
    extended_text = generate(model, tokenizer, prompt, max_new_tokens=10)
    answer = extended_text[len(prompt):]
    return answer.strip().splitlines()[0].strip()

Let's test this function:

In [ ]:
get_capital_city(gpt2, gpt2_tokenizer, "United Kingdom")

In [ ]:
get_capital_city(gpt2, gpt2_tokenizer, "Mexico")

It works! And it's pretty flexible with its inputs, too:

In [ ]:
[get_capital_city(gpt2, gpt2_tokenizer, country)
 for country in ("The UK", "Great Britain", "Big Britane")]

However, it's far from perfect. In particular, it picked up many misconceptions from its training data. The capital cities of the following countries are respectively Canberra, Ottawa, Wellington, and Washington D.C., but GPT-2 gets it wrong:

In [ ]:
[get_capital_city(gpt2, gpt2_tokenizer, country)
 for country in ("Australia", "Canada", "New Zealand", "USA")]

And when the model is not sure, it often returns the name of the country (which is only the correct answer for about 10 countries).

In [ ]:
[get_capital_city(gpt2, gpt2_tokenizer, country)
 for country in ("Buthan", "Colombia", "Togo")]

And when the input is not a country at all, the model often defaults to Paris, which is the only example it saw in its prompt:

In [ ]:
[get_capital_city(gpt2, gpt2_tokenizer, country)
 for country in ("hey", "yo", "j")]

In [ ]:
del model_id, gpt2
free_vram(device)

In [ ]:
if IS_COLAB:
    from google.colab import userdata

    access_token = userdata.get('token-hf-read-mistral')
else:
    # if you are not running this notebook in Colab, then you can
    # save your access token in a file named hf-read-mistral.secret in this
    # notebook's directory (keep it safe!), and load the secret using this code:
    access_token = open("hf-read-mistral.secret").read().strip()

    # Alternatively, you can simply hard-code your access token like this:
    # access_token = "..."  # WARNING: hard-coding secrets is very insecure

In [ ]:
from huggingface_hub import login

login(access_token)

In [ ]:
model_id = "mistralai/Mistral-7B-v0.3"
mistral7b_tokenizer = AutoTokenizer.from_pretrained(model_id)
mistral7b = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", dtype="auto")

In [ ]:
torch.manual_seed(42)
generate(mistral7b, mistral7b_tokenizer, prompt, do_sample=True, top_p=0.6)

This model is much more knowledgeable, it gets almost all capitals right.

In [ ]:
[get_capital_city(mistral7b, mistral7b_tokenizer, country)
 for country in ("Australia", "Canada", "New Zealand", "USA")]

And when the model is not sure, it often returns the name of the country (which is only the correct answer for about 10 countries).

In [ ]:
[get_capital_city(mistral7b, mistral7b_tokenizer, country)
 for country in ("Buthan", "Colombia", "Togo")]

Can we chat with it?

In [ ]:
prompt = "List some places I should visit in Paris."
generate(mistral7b, mistral7b_tokenizer, prompt)

In [ ]:
bob_introduction = """
Bob is an amazing chatbot. It knows everything and it's incredibly helpful.
"""

In [ ]:
full_prompt = f"{bob_introduction}Me: {prompt}\nBob:"
extended_text = generate(mistral7b, mistral7b_tokenizer, full_prompt,
                         max_new_tokens=100)
answer = extended_text[len(full_prompt):].strip()
print(answer)

In [ ]:
answer.split("\nMe: ")[0]

In [ ]:
class BobTheChatbot:  # or ChatBob if you prefer
    def __init__(self, model, tokenizer, introduction=bob_introduction,
                 max_answer_length=10_000):
        self.model = model
        self.tokenizer = tokenizer
        self.context = introduction
        self.max_answer_length = max_answer_length

    def chat(self, prompt):
        self.context += "\nMe: " + prompt + "\nBob:"
        context = self.context
        start_index = len(context)
        while True:
            extended = generate(self.model, self.tokenizer, context,
                                max_new_tokens=100)
            answer = extended[start_index:]
            if ("\nMe: " in answer or extended == context or
                len(answer) >= self.max_answer_length): break
            context = extended
        answer = answer.split("\nMe: ")[0]
        self.context += answer
        return answer.strip()

In [ ]:
bob = BobTheChatbot(mistral7b, mistral7b_tokenizer)
bob.chat("List some places I should visit in Paris.")

In [ ]:
bob.chat("Tell me more about the first place.")

In [ ]:
bob.chat("And Rome?")

Great! We wrote a working chatbot in about 20 lines of code!

However, Bob is not perfect. It can struggle with repetition, falling into loops:

In [ ]:
bob = BobTheChatbot(mistral7b, mistral7b_tokenizer)
print(bob.chat("Tell me 5 jokes"))

It's not always sufficiently precise or helpful:

In [ ]:
bob = BobTheChatbot(mistral7b, mistral7b_tokenizer)
print(bob.chat("How can I make cookies?"))

And Bob can be a bad boy:

In [ ]:
bad_bob = BobTheChatbot(mistral7b, mistral7b_tokenizer)
bad_bob.chat("I'd like to rob a bank. How should I prepare?")

We need to fine-tune the model to make it more conversational and helpful. We can do this by fine-tuning it on a dataset containing the type of dialogue we'd like to see. This is typically done in two stages: SFT + DPO. The first is just regular supervised fine-tuning except we only compute the loss based on the model's answer. We will see an example using the TRL library shortly. DPO, is a bit more difficult (and interesting!), so let's look into it.

## DPO

In [ ]:
prompt = "The capital of Argentina is "
full_input = [prompt + "Buenos Aires", prompt + "Madrid"]
mistral7b_tokenizer.pad_token = mistral7b_tokenizer.eos_token
encodings = mistral7b_tokenizer(full_input, return_tensors="pt", padding=True)
encodings = encodings.to(device)
with torch.no_grad():
  logits = mistral7b(**encodings).logits  # shape [2, 8, 32768]

In [ ]:
encodings.input_ids

* Token #1 = start-of-sequence
* Token #2 = end-of-sequence & padding token

In [ ]:
next_token_ids = encodings.input_ids[:, 1:]  # shape [2, 7]
log_probas = F.log_softmax(logits, dim=-1)[:, :-1]  # shape [2, 7, 32768]
next_token_log_probas = torch.gather(  # shape [2, 7, 1]
    log_probas, dim=2, index=next_token_ids.unsqueeze(2))

Alternatively:

In [ ]:
next_token_log_probas = -F.cross_entropy(
    logits[:, :-1].permute(0, 2, 1), next_token_ids, reduction="none")

Log probas for every token in "The capital of Argentina is Buenos Aires":

In [ ]:
[f"{p.item():.2%}" for p in torch.exp(next_token_log_probas[0])]

The token "The" has a 3.32% proba of following the first token (start-of-sequence).
Then the token "capital" has a 0.02% proba of being the next token after that. And so on.

Now the log probas for "The capital of Argentina is Madrid" (the first token is the padding token):

In [ ]:
[f"{p.item():.2%}" for p in torch.exp(next_token_log_probas[1])]

In [ ]:
# extra code – shows the top 10 next tokens after token #5 in the first
# sentence (i.e., token "is")
token_index = 5
topk = torch.topk(log_probas[0, token_index], k=10)
for v, i in zip(topk.values, topk.indices):
    print(f"{torch.exp(v).item():06.2%} {mistral7b_tokenizer.decode([i])}")

In [ ]:
answer_log_proba = next_token_log_probas[0, -2:].sum()  # Buenos + Aires
torch.exp(answer_log_proba).item()  # proba of "Buenos Aires" given the rest

In [ ]:
padding_mask = encodings.attention_mask[:, :-1]
log_probas_sum = (next_token_log_probas * padding_mask).sum(dim=1)
log_probas_sum

In [ ]:
def sum_of_log_probas(model, tokenizer, full_inputs):
    encodings = tokenizer(
        full_inputs, return_tensors="pt", padding=True).to(model.device)
    logits = model(**encodings).logits
    next_token_log_probas = -F.cross_entropy(
        logits[:, :-1].permute(0, 2, 1), encodings.input_ids[:, 1:],
        reduction="none")
    return (next_token_log_probas * encodings.attention_mask[:, :-1]).sum(dim=1)

def dpo_loss(model, ref_model, tokenizer, full_input_c, full_input_r, beta=0.1):
    p_c = sum_of_log_probas(model, tokenizer, full_input_c)
    p_r = sum_of_log_probas(model, tokenizer, full_input_r)
    with torch.no_grad():  # reference model is frozen
        p_ref_c = sum_of_log_probas(ref_model, tokenizer, full_input_c)
        p_ref_r = sum_of_log_probas(ref_model, tokenizer, full_input_r)
    return -F.logsigmoid(beta*((p_c - p_ref_c) - (p_r - p_ref_r))).mean()

In [ ]:
with torch.no_grad():
  result = sum_of_log_probas(mistral7b, mistral7b_tokenizer, full_input)

result

In [ ]:
with torch.no_grad():
  loss = dpo_loss(mistral7b, mistral7b, mistral7b_tokenizer,
                  [full_input[0]], [full_input[1]])

loss

Let's clean up the global namespace and free some GPU RAM:

In [ ]:
del_vars(["access_token", "answer", "answer_log_proba", "bad_bob", "bob",
          "bob_introduction", "encodings", "extended_text", "full_input",
          "full_prompt", "i", "log_probas", "log_probas_sum", "logits",
          "mistral7b", "mistral7b_tokenizer", "model_id", "next_token_ids",
          "next_token_log_probas", "padding_mask", "prompt", "token_index",
          "topk", "v"])

## Fine-Tuning a model using the TRL library

### Fine-tuning GPT2 with SFT on the Alpaca dataset

In [ ]:
sft_dataset = load_dataset("tatsu-lab/alpaca", split="train")

In [ ]:
print(sft_dataset[1]["text"])

In [ ]:
def preprocess(example):
    text = f"Human: {example['instruction']}\n"
    if example['input'] != "":
        text += f"-> {example['input']}\n"
    text += f"\nAssistant: {example['output']}"
    return {"text": text}

sft_dataset = sft_dataset.map(preprocess)

In [ ]:
print(sft_dataset[1]["text"])

In [ ]:
# extra code – downsize the dataset to just 1,000 instances
# because this is just a quick demo
sft_dataset = sft_dataset.shuffle(seed=42).select(range(1000))

If running this notebook on Colab or Kaggle, the trl library is not preinstalled, so we must install it manually:

In [ ]:
if IS_COLAB or IS_KAGGLE:
    %pip install -qU trl

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_model_dir = "./my_gpt2_sft_alpaca"
training_args = SFTConfig(
    output_dir=sft_model_dir, max_length=512,
    per_device_train_batch_size=4, num_train_epochs=1, save_steps=50,
    logging_steps=10, learning_rate=5e-5, report_to="none")
sft_trainer = SFTTrainer("gpt2", train_dataset=sft_dataset, args=training_args)
sft_train_output = sft_trainer.train()
sft_trainer.model.save_pretrained(sft_model_dir)

In [ ]:
del sft_dataset, training_args, sft_trainer, sft_train_output
free_vram(device)

### Fine-tuning with DPO on the ministrack-preferences dataset

In [ ]:
pref_dataset = load_dataset("Anthropic/hh-rlhf", split="train")

In [ ]:
pref_dataset[2].keys()

In [ ]:
print(pref_dataset[2]["chosen"].strip())

In [ ]:
print(pref_dataset[2]["rejected"].strip())

In [ ]:
# extra code – downsize the dataset to just 1,00 instances
# because this is just a quick demo
pref_dataset = pref_dataset.shuffle(seed=42).select(range(100))

In [ ]:
from trl import DPOConfig, DPOTrainer

dpo_model_dir = "./my_gpt2_sft_alpaca_dpo_hh_rlhf"
training_args = DPOConfig(
    output_dir=dpo_model_dir, max_length=512, per_device_train_batch_size=4,
    num_train_epochs=1, save_steps=50, logging_steps=10, learning_rate=2e-5,
    report_to="none")
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
dpo_trainer = DPOTrainer(
    sft_model_dir, args=training_args, train_dataset=pref_dataset,
    processing_class=gpt2_tokenizer)
dpo_train_output = dpo_trainer.train()
dpo_trainer.model.save_pretrained(dpo_model_dir)

Let's free some GPU RAM:

In [ ]:
del_vars(["dpo_model_dir", "dpo_train_output", "dpo_trainer", "gpt2_tokenizer",
          "pref_dataset", "sft_model_dir", "training_args"])

## Off-the-shelf chatbot models

Instead of fine-tuning your own chatbot model, you can download one directly, already pretrained then fine-tuned for improved conversational and instruction-following skills. For example, let's use Mistral-7B-Instruct-v0.3:

In [ ]:
model_id = "mistralai/Mistral-7B-Instruct-v0.3"
mistral7bi_tokenizer = AutoTokenizer.from_pretrained(model_id)
mistral7bi = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", dtype="auto")

Bob now tells 5 _different_ jokes:

In [ ]:
good_bob = BobTheChatbot(mistral7bi, mistral7bi_tokenizer)
print(good_bob.chat("Tell me 5 jokes"))

Bob makes great cookies:

In [ ]:
good_bob = BobTheChatbot(mistral7bi, mistral7bi_tokenizer)
print(good_bob.chat("How can I make cookies?"))

Last but not least, Bob is a good boy now.

In [ ]:
good_bob = BobTheChatbot(mistral7bi, mistral7bi_tokenizer)
good_bob.chat("I'd like to rob a bank. How should I prepare?")

In [ ]:
del good_bob, mistral7bi, mistral7bi_tokenizer, model_id
free_vram(device)

# Extra Material — Fixed Positional Encodings
The authors of the Transformer paper proposed a method to replace the trainable positional embeddings with fixed positional encodings, using a scheme based on the sine and cosine functions at different frequencies. The positional encoding matrix **P** is defined using the following equation:

$
\mathrm{PE}(p,i) = \left\{\begin{array}{left center} \sin (p/ 10000^{i/ d}) & \text{if }i\text{ is even} \\ \cos (p/ 10000^{(i - 1)/ d}) & \text{if }i\text{ is odd} \\  \end{array}\right.
$

$\mathrm{PE}(p,i)$ is the $i$<sup>th</sup> component of the encoding for the token located at the $p$<sup>th</sup> position in the sentence.

Here is how it can be implemented in PyTorch:

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, max_length, embed_dim, dropout=0.1):
        super().__init__()
        p = torch.arange(max_length).unsqueeze(1)  # shape: [max_length, 1]
        i = torch.arange(0, embed_dim, 2)  # shape: [embed_dim // 2]
        angle = p / 10_000 ** (i / embed_dim)  # [max_length, embed_dim // 2]
        pos_encodings = torch.empty(max_length, embed_dim)
        pos_encodings[:, ::2] = angle.sin()
        pos_encodings[:, 1::2] = angle.cos()
        self.register_buffer("pos_encodings", pos_encodings)
        self.dropout = nn.Dropout(dropout)

    def forward(self, X):  # X shape: [batch_size, max_batch_length, embed_dim]
        return self.dropout(X + self.pos_encodings[:X.size(1)])

It can then be used like so:

In [ ]:
max_length = 500
embed_dim = 512
pos_encoding = PositionalEncoding(max_length, embed_dim)
embeddings = torch.randn(256, 500, 512)
embeddings_with_pos = pos_encoding(embeddings)
embeddings_with_pos.shape

Now let's plot the positional encoding matrix **P**:

In [ ]:
figure_max_length = 201
figure_enc_size = 512
pos_enc = PositionalEncoding(figure_max_length, figure_enc_size)
P = pos_enc.pos_encodings.numpy()
i1, i2, crop_i = 100, 101, 150
p1, p2, p3 = 22, 60, 35

fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, sharex=True, figsize=(9, 5))
img = ax1.imshow(P.T[:crop_i], cmap="gray", interpolation="bilinear", aspect="auto")
cheat = 2  # need to raise the red line a bit, or else it hides the blue one
ax1.set_title("Sine/cosine positional encoding matrix (transposed)")
ax1.hlines(i2+cheat, 0, figure_max_length - 1, color="r", linewidth=3)
ax1.hlines(i1, 0, figure_max_length - 1, color="b", linewidth=3)
ax1.plot([p1, p1], [0, crop_i], "k--")
ax1.plot([p2, p2], [0, crop_i], "k--", alpha=0.5)
ax1.plot([p1, p2], [i2+cheat, i2+cheat], "ro")
ax1.plot([p1, p2], [i1, i1], "bo")
ax1.axis([0, figure_max_length - 1, 0, crop_i])
ax1.set_ylabel("Dim. $i$", rotation=0, fontsize=12)

ax2.set_title("Zoom on $i = 100$ and $i = 101$")
ax2.plot([p1, p1], [-1, 1], "k--", label="$p = {}$".format(p1))
ax2.plot([p2, p2], [-1, 1], "k--", label="$p = {}$".format(p2), alpha=0.5)
ax2.plot(p3, P[p3, i1], "bx", label="$p = {}$".format(p3))
ax2.plot(P[:,i1], "b-", label="$i = {}$".format(i1))
ax2.plot(P[:,i2], "r-", label="$i = {}$".format(i2))
ax2.plot([p1, p2], [P[p1, i1], P[p2, i1]], "bo")
ax2.plot([p1, p2], [P[p1, i2], P[p2, i2]], "ro")
ax2.set_xlabel("Token position $p$", fontsize=12)
ax2.legend(loc="center right", fontsize=14, framealpha=0.95)
ax2.set_ylabel("$PE(p,i)$", rotation=0, fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.hlines(0, 0, figure_max_length - 1, color="k", linewidth=1, alpha=0.3)
ax2.axis([0, figure_max_length - 1, -1, 1])
plt.show()

Once these positional encodings are added to the token embeddings, the model has access to the absolute position of each token in the sentence because there is a unique positional encoding for each position (e.g., the positional encoding for the token located at the 22nd position in a sentence is represented by the vertical dashed line at the top left of the figure, and you can see that it's unique to that position).

Moreover, the choice of oscillating functions (sine and cosine) makes it possible for the model to easily learn relative positions as well. For example, two tokens located $\Delta = 38$ tokens apart (e.g., at positions _p_ = 22 and _p_ = 60) always have the same positional encoding values in the encoding dimensions _i_ = 100 and _i_ = 101. And that's not all: given a relative distance $\Delta$, it's always possible to compute $\cos(x + \Delta)$ and $\sin(x + \Delta)$ using a linear function of $\cos(x)$ and $\sin(x)$, so when the model learns a pattern at some position, it can easily generalize it to other positions:

$$
\begin{array}
\sin(x+\Delta) = \sin(x)\cos(\Delta) + \cos(x)\sin(\Delta) \\
\cos(x+\Delta) = \cos(x)\cos(\Delta) - \sin(x)\sin(\Delta)
\end{array}
$$

You might wonder why we can't simply add the token index as an extra dimension in each token's representation. Well, it would certainly be better than nothing, but it wouldn't work as well as the sine/cosine approach:
* sine and cosine are bounded between –1 and +1, while the token index would not be bounded.
* with sine/cosine, the model can easily learn patterns in relative positioning.
* thanks to the many frequencies, it's easier for the model to learn patterns at various scales.

The main advantage of fixed positional encodings over trainable positional embeddings is that they don't add any model parameters. This can be beneficial when dealing with fairly small datasets. For large ones, however, trainable positional embeddings are usually preferred. Or even better: use modern positional encodings such as RoPE or ALiBi (see Chapter 17).

Let's free some GPU RAM:

In [ ]:
del_vars(["max_length", "embed_dim", "pos_encoding", "embeddings",
          "embeddings_with_pos", "figure_max_length", "figure_enc_size",
          "pos_enc", "P", "i1", "i2", "crop_i", "p1", "p2", "p3", "fig",
          "ax1", "ax2", "img", "cheat"])

# Exercise solutions

## Work in progress

I'm working on the exercise solutions, hoping to finish them by December 2025. Thanks for your patience!